# Talk to Government Data

A Natural Language Analyst over Open Public Data

Author: Hritviz Manral

## Problem Statement
Ask questions in natural language and get data-backed answers, charts, provenance, confidence scores, and refusal handling.

In [ ]:
!pip install openai pandas matplotlib gradio

## Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('Air_Quality.csv')
df['last_update'] = pd.to_datetime(df['last_update'])
df.head()

## Architecture
Question → GPT-4o-mini → Structured JSON → Validator → Query Engine → Chart Engine → Confidence Engine → Answer

In [ ]:
OPENAI_API_KEY = 'YOUR_API_KEY'

## Parse_question() below

In [ ]:
import json
from openai import OpenAI

from config import OPENAI_API_KEY

client = OpenAI(api_key=OPENAI_API_KEY)

def load_system_prompt():

    with open(
        "prompts/parser_prompt.txt",
        "r",
        encoding="utf-8"
    ) as f:

        return f.read()
    

def parse_question(question):

    system_prompt = load_system_prompt()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    content = response.choices[0].message.content

    return json.loads(content)

## Validator.py below

In [ ]:
SUPPORTED_INTENTS = {
    "average",
    "maximum",
    "minimum",
    "compare",
    "group_compare",
    "top_n",
    "bottom_n",
    "distribution",
    "count",
    "unique_values",
    "station_lookup",
    "location_filter",
    "summary",
    "refuse"
}


def validate_query(query_json):

    if "intent" not in query_json:
        raise ValueError("Missing intent")

    intent = query_json["intent"]

    if intent not in SUPPORTED_INTENTS:
        raise ValueError(f"Unsupported intent: {intent}")

    return True

## QueryEngine below

In [ ]:
import pandas as pd


class QueryEngine:

    

    def __init__(self, df):
        self.df = df

    def apply_filters(self, df, filters):

        if not filters:
            return df

        for column, value in filters.items():

            if value is None:
                continue

            df = df[
                df[column]
                .astype(str)
                .str.lower()
                ==
                str(value).lower()
            ]

        return df
    
    def clean_pollutant_data(self, df):

        return df.dropna(
            subset=["pollutant_avg"]
        )

    def average(self, query):

        print("========== NEW QUERY ENGINE ==========")

        pollutant = query["pollutant"]

        df = self.df

        df = df[
            df["pollutant_id"].str.upper()
            ==
            pollutant.upper()
        ]

        df = self.apply_filters(
            df,
            query.get("filters", {})
        )

        df = self.clean_pollutant_data(df)

        value = df["pollutant_avg"].mean()

        return {
            "answer": f"Average {pollutant} is {value:.2f}",

            "data": {
                "average": float(value)
            },

            "provenance": {
                "intent": "average",
                "rows_used": len(df),
                "filters": query.get("filters", {}),
                "pollutant": pollutant
            },

            "chart_data": None
        }

    def maximum(self, query):

        pollutant = query["pollutant"]

        df = self.df

        df = df[
            df["pollutant_id"].str.upper()
            ==
            pollutant.upper()
        ]

        df = self.clean_pollutant_data(df)

        if df.empty:
            return {
                "answer": f"No data found for {pollutant}",
                "data": None,
                "provenance": {},
                "chart_data": None
            }

        idx = df["pollutant_avg"].idxmax()

        row = df.loc[idx]

        return {
            "answer":
                f"Highest {pollutant} found in "
                f"{row['city']} "
                f"with value "
                f"{row['pollutant_avg']}",

            "data": row.to_dict(),

            "provenance": {
                "intent": "maximum",
                "rows_used": len(df),
                "pollutant": pollutant
            },

            "chart_data": None
        }

    def count(self, query):

        target = query.get(
            "target",
            "records"
        )

        if target == "station":

            value = self.df[
                "station"
            ].nunique()

        else:

            value = len(self.df)

        return {
            "answer": f"Count is {value}",

            "data": {
                "count": int(value)
            },

            "provenance": {
                "intent": "count",
                "target": target
            },

            "chart_data": None
        }

    def compare(self, query):

        pollutant = query["pollutant"]

        cities = query["cities"]

        df = self.df

        df = df[
            df["pollutant_id"].str.upper()
            ==
            pollutant.upper()
        ]

        result = []

        for city in cities:

            city_df = df[
                df["city"].str.lower()
                ==
                city.lower()
            ]

            city_df = self.clean_pollutant_data(
                city_df
            )

            avg = city_df["pollutant_avg"].mean()

            result.append({
                "city": city,
                "value": float(avg)
            })

        return {
            "answer":
                f"Comparison completed for {pollutant}",

            "data": result,

            "provenance": {
                "intent": "compare",
                "pollutant": pollutant,
                "cities": cities
            },

            "chart_data": result
        }

    def minimum(self, query):

        df = self.df.copy()

        pollutant = query.get("pollutant")

        if pollutant:
            df = df[
                df["pollutant_id"].str.upper()
                ==
                pollutant.upper()
            ]

        df = self.apply_filters(
            df,
            query.get("filters", {})
        )

        if df.empty:
            return {
                "answer": "No matching data found",
                "data": None,
                "provenance": {
                    "intent": "minimum"
                },
                "chart_data": None
            }
        
        df = self.clean_pollutant_data(df)

        minimum = float(
            df["pollutant_avg"].min()
        )

        return {
            "answer":
                f"Minimum {pollutant} is "
                f"{minimum:.2f}",

            "data": {
                "minimum": minimum
            },

            "provenance": {
                "intent": "minimum",
                "rows_used": len(df),
                "filters":
                    query.get("filters", {}),
                "pollutant": pollutant
            },

            "chart_data": None
        }
    
    def top_n(self, query):

        pollutant = query["pollutant"]

        group_by = query["group_by"]

        n = query.get("n", 5)

        df = self.df.copy()

        df = df[
            df["pollutant_id"].str.upper()
            ==
            pollutant.upper()
        ]

        df = self.apply_filters(
            df,
            query.get("filters", {})
        )

        df = self.clean_pollutant_data(df)

        if df.empty:
            return {
                "answer": "No matching data found",
                "data": None,
                "provenance": {},
                "chart_data": None
            }

        result = (
            df.groupby(group_by)["pollutant_avg"]
            .mean()
            .sort_values(ascending=False)
            .head(n)
            .reset_index()
        )

        records = result.to_dict(
            orient="records"
        )

        return {
            "answer":
                f"Top {n} {group_by}s by {pollutant}",

            "data": records,

            "provenance": {
                "intent": "top_n",
                "rows_used": len(df),
                "group_by": group_by,
                "pollutant": pollutant
            },

            "chart_data": records
        }
    
    def bottom_n(self, query):

        pollutant = query["pollutant"]

        group_by = query["group_by"]

        n = query.get("n", 5)

        df = self.df.copy()

        df = df[
            df["pollutant_id"].str.upper()
            ==
            pollutant.upper()
        ]

        df = self.apply_filters(
            df,
            query.get("filters", {})
        )

        df = self.clean_pollutant_data(df)

        if df.empty:
            return {
                "answer": "No matching data found",
                "data": None,
                "provenance": {},
                "chart_data": None
            }

        result = (
            df.groupby(group_by)["pollutant_avg"]
            .mean()
            .sort_values(ascending=True)
            .head(n)
            .reset_index()
        )

        records = result.to_dict(
            orient="records"
        )

        return {
            "answer":
                f"Bottom {n} {group_by}s by {pollutant}",

            "data": records,

            "provenance": {
                "intent": "bottom_n",
                "rows_used": len(df),
                "group_by": group_by,
                "pollutant": pollutant
            },

            "chart_data": records
        }

    def summary(self, query):

        df = self.df.copy()

        pollutant = query.get("pollutant")

        if pollutant:
            df = df[
                df["pollutant_id"].str.upper()
                ==
                pollutant.upper()
            ]

        df = self.apply_filters(
            df,
            query.get("filters", {})
        )

        df = self.clean_pollutant_data(df)

        if df.empty:
            return {
                "answer": "No matching data found",
                "data": None,
                "provenance": {},
                "chart_data": None
            }

        stats = {
            "count": int(len(df)),
            "mean": float(df["pollutant_avg"].mean()),
            "median": float(df["pollutant_avg"].median()),
            "min": float(df["pollutant_avg"].min()),
            "max": float(df["pollutant_avg"].max())
        }

        return {
            "answer": f"Summary statistics for {pollutant}",
            "data": stats,
            "provenance": {
                "intent": "summary",
                "rows_used": len(df),
                "pollutant": pollutant,
                "filters": query.get("filters", {})
            },
            "chart_data": None
        }


    def unique_values(self, query):

        column = query["column"]

        values = sorted(
            self.df[column]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        return {
            "answer":
                f"Found {len(values)} unique values in {column}",
            "data": values,
            "provenance": {
                "intent": "unique_values",
                "column": column
            },
            "chart_data": None
        }


    def station_lookup(self, query):

        station = query["station"]

        df = self.df[
            self.df["station"]
            .astype(str)
            .str.contains(
            station,
            case=False,
            na=False,
            regex=False
            )
        ]

        if df.empty:
            return {
                "answer": f"No station found matching {station}",
                "data": None,
                "provenance": {
                    "intent": "station_lookup"
                },
                "chart_data": None
            }

        rows = df.head(20).to_dict(
            orient="records"
        )

        return {
            "answer":
                f"Found {len(df)} records for station {station}",
            "data": rows,
            "provenance": {
                "intent": "station_lookup",
                "station": station
            },
            "chart_data": None
        }


    def location_filter(self, query):

        df = self.df.copy()

        pollutant = query.get("pollutant")

        if pollutant:
            df = df[
                df["pollutant_id"].str.upper()
                ==
                pollutant.upper()
            ]

        df = self.apply_filters(
            df,
            query.get("filters", {})
        )

        rows = df.head(100).to_dict(
            orient="records"
        )

        return {
            "answer":
                f"Found {len(df)} matching rows",
            "data": rows,
            "provenance": {
                "intent": "location_filter",
                "rows_used": len(df),
                "filters": query.get("filters", {})
            },
            "chart_data": None
        }


    def distribution(self, query):

        pollutant = query["pollutant"]

        df = self.df.copy()

        df = df[
            df["pollutant_id"].str.upper()
            ==
            pollutant.upper()
        ]

        df = self.apply_filters(
            df,
            query.get("filters", {})
        )

        df = self.clean_pollutant_data(df)

        values = (
            df["pollutant_avg"]
            .dropna()
            .tolist()
        )

        return {
            "answer":
                f"Distribution for {pollutant}",
            "data": {
                "values": values
            },
            "provenance": {
                "intent": "distribution",
                "rows_used": len(df),
                "pollutant": pollutant
            },
            "chart_data": {
                "type": "histogram",
                "values": values
            }
        }

    def execute(self, query):

        intent = query["intent"]

        if intent == "average":
            return self.average(query)

        elif intent == "maximum":
            return self.maximum(query)

        elif intent == "minimum":
            return self.minimum(query)

        elif intent == "count":
            return self.count(query)

        elif intent == "compare":
            return self.compare(query)
        
        elif intent == "top_n":
            return self.top_n(query)
        
        elif intent == "bottom_n":
            return self.bottom_n(query)
        
        elif intent == "summary":
            return self.summary(query)

        elif intent == "unique_values":
            return self.unique_values(query)

        elif intent == "station_lookup":
            return self.station_lookup(query)

        elif intent == "location_filter":
            return self.location_filter(query)

        elif intent == "distribution":
            return self.distribution(query)

        elif intent == "refuse":
            return {
                "answer":
                    query.get(
                        "reason",
                        "Cannot answer from dataset"
                    ),

                "data": None,

                "provenance": {
                    "intent": "refuse"
                },

                "chart_data": None
            }
        

        raise ValueError(
            f"Intent not implemented: {intent}"
        )

## ChartEngine below

In [ ]:
import os
import uuid

import matplotlib.pyplot as plt


class ChartEngine:

    def __init__(self):

        self.output_dir = "charts"

        os.makedirs(
            self.output_dir,
            exist_ok=True
        )

    def create_bar_chart(
        self,
        data,
        x_col,
        y_col,
        title
    ):

        plt.figure(
            figsize=(8, 5)
        )

        x = [
            row[x_col]
            for row in data
        ]

        y = [
            row[y_col]
            for row in data
        ]

        plt.bar(x, y)

        plt.title(title)

        plt.xticks(
            rotation=45,
            ha="right"
        )

        plt.tight_layout()

        filename = (
            f"{uuid.uuid4()}.png"
        )

        path = os.path.join(
            self.output_dir,
            filename
        )

        plt.savefig(path)

        plt.close()

        return path

    def create_histogram(
        self,
        values,
        title
    ):

        plt.figure(
            figsize=(8, 5)
        )

        plt.hist(
            values,
            bins=20
        )

        plt.title(title)

        plt.tight_layout()

        filename = (
            f"{uuid.uuid4()}.png"
        )

        path = os.path.join(
            self.output_dir,
            filename
        )

        plt.savefig(path)

        plt.close()

        return path

## ConfidenceEngine below

In [ ]:
class ConfidenceEngine:

    def calculate(self, query, result):

        confidence = 1.0

        rows_used = (
            result["provenance"]
            .get("rows_used", 0)
        )

        if rows_used < 20:
            confidence -= 0.10

        if rows_used < 10:
            confidence -= 0.15

        if rows_used < 5:
            confidence -= 0.25

        if query["intent"] in [
            "station_lookup",
            "location_filter"
        ]:
            confidence -= 0.05

        confidence = max(
            0.0,
            min(confidence, 1.0)
        )

        return round(
            confidence,
            2
        )

## DataPipeline below

In [ ]:
from engine.query_engine import QueryEngine
from engine.chart_engine import ChartEngine
from engine.confidence_engine import ConfidenceEngine

class DataPipeline:

    def __init__(self, df):

        self.query_engine = QueryEngine(df)

        self.chart_engine = ChartEngine()

        self.confidence_engine = (
            ConfidenceEngine()
        )

    def run(self, query):

        result = self.query_engine.execute(
            query
        )

        chart_data = result.get(
            "chart_data"
        )

        chart_path = None

        if chart_data:

            intent = query["intent"]

            if intent in [
                "top_n",
                "bottom_n"
            ]:

                group_by = query[
                    "group_by"
                ]

                chart_path = (
                    self.chart_engine
                    .create_bar_chart(
                        data=chart_data,
                        x_col=group_by,
                        y_col="pollutant_avg",
                        title=result["answer"]
                    )
                )

            elif intent == "compare":

                chart_path = (
                    self.chart_engine
                    .create_bar_chart(
                        data=chart_data,
                        x_col="city",
                        y_col="value",
                        title=result["answer"]
                    )
                )

            elif intent == "distribution":

                chart_path = (
                    self.chart_engine
                    .create_histogram(
                        values=chart_data[
                            "values"
                        ],
                        title=result["answer"]
                    )
                )

        result["chart_path"] = chart_path

        confidence = (
            self.confidence_engine
            .calculate(
            query,
            result
        )
        )

        result["confidence"] = (
            confidence
        )

        result["needs_human_review"] = (
            confidence < 0.70
        )

        return result

In [ ]:
pipeline = DataPipeline(df)

In [ ]:
query = {
    'intent':'average',
    'pollutant':'PM2.5',
    'filters':{'state':'Bihar'}
}

pipeline.run(query)

## Evaluation Results
PASS: average, maximum, minimum, count, compare, top_n, bottom_n, summary, unique_values, station_lookup, location_filter, distribution, refusal handling, confidence scoring

## Future Work
- Agentic self-correction
- Multi-dataset support
- DuckDB backend
- Confidence calibration

## Final Gradio app.py code below and use demo.launch(share=True)

In [ ]:
import gradio as gr

from utils.data_loader import load_data
from utils.explanation import generate_explanation

from llm.openai_parser import parse_question

from pipeline import DataPipeline


# Load dataset once
df = load_data()

# Create pipeline once
pipeline = DataPipeline(df)


def chat(question):

    try:

        if not question.strip():

            return (
                {},
                "No question provided.",
                "Please enter a question.",
                {},
                None,
                0.0,
                "Needs Human Review"
            )

        query = parse_question(
            question
        )

        # Refusal Handling
        if query.get("intent") == "refuse":

            return (
                query,
                "The query cannot be answered from the available dataset.",
                query.get(
                    "reason",
                    "Question cannot be answered from dataset."
                ),
                {
                    "intent": "refuse"
                },
                None,
                0.0,
                "Needs Human Review"
            )

        result = pipeline.run(
            query
        )

        explanation = (
            generate_explanation(
                query
            )
        )

        return (
            query,
            explanation,
            result["answer"],
            result["provenance"],
            result.get(
                "chart_path"
            ),
            result.get(
                "confidence",
                1.0
            ),
            (
                "Needs Human Review"
                if result.get(
                    "needs_human_review",
                    False
                )
                else
                "Approved"
            )
        )

    except Exception as e:

        return (
            {
                "error": str(e)
            },
            "Pipeline execution failed.",
            str(e),
            {},
            None,
            0.0,
            "Needs Human Review"
        )


with gr.Blocks(
    title="Talk to Government Data"
) as demo:

    gr.Markdown(
        """
# Talk to Government Data

Ask questions about India's Air Quality dataset using natural language.

The system:
- Converts questions into structured JSON
- Executes real computations on the dataset
- Generates charts automatically
- Shows provenance
- Refuses out-of-scope questions
"""
    )

    with gr.Row():

        question_box = gr.Textbox(
            label="Ask a Question",
            placeholder="Example: Top 5 cities by PM2.5",
            lines=2,
            scale=8
        )

        ask_button = gr.Button(
            "Ask",
            scale=1
        )

    gr.Examples(
        examples=[
            ["Average PM2.5 in Bihar"],
            ["Maximum PM10 in Delhi"],
            ["Top 5 cities by PM2.5"],
            ["Bottom 10 stations by PM10"],
            ["Summarize PM2.5 in Delhi"],
            ["Show distribution of PM2.5"],
            ["Which states exist?"],
            ["Show Anand Vihar station"],
            ["Who is the Prime Minister of India?"]
        ],
        inputs=question_box
    )

    with gr.Row():

        with gr.Column():

            generated_json = gr.JSON(
                label="Generated JSON"
            )

            provenance_box = gr.JSON(
                label="Provenance"
            )

        with gr.Column():

            explanation_box = gr.Textbox(
                label="Query Explanation",
                lines=5
            )

            answer_box = gr.Textbox(
                label="Answer",
                lines=3
            )

    with gr.Row():

        confidence_box = gr.Number(
            label="Confidence Score"
        )

        review_box = gr.Textbox(
            label="Review Status"
        )

    chart_box = gr.Image(
        label="Generated Chart"
    )

    ask_button.click(
        fn=chat,
        inputs=question_box,
        outputs=[
            generated_json,
            explanation_box,
            answer_box,
            provenance_box,
            chart_box,
            confidence_box,
            review_box
        ]
    )

    question_box.submit(
        fn=chat,
        inputs=question_box,
        outputs=[
            generated_json,
            explanation_box,
            answer_box,
            provenance_box,
            chart_box,
            confidence_box,
            review_box
        ]
    )


if __name__ == "__main__":

    demo.launch(
    share=True
    )